### Running this notebook

**The visualizations in this notebook will run in [jupyter lab](https://github.com/jupyterlab/jupyterlab#installation), not jupyter notebook. Google colab is not supported either. VS Code notebooks _might_ work but that has not been tested.** See the fastplotlib supported frameworks for more info: https://github.com/fastplotlib/fastplotlib/#supported-frameworks 

In [ ]:
import os
import glob
from copy import deepcopy

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import mesmerize_core as mc
import mesmerize_viz
import fastplotlib as fpl
import concat_tif as ct

from mesmerize_core.caiman_extensions.cnmf import cnmf_cache

from pathlib import Path

if os.name == "nt":
    # disable the cache on windows, this will be automatic in a future version
    cnmf_cache.set_maxsize(0)

pd.options.display.max_colwidth = 120

## Set paths

In [ ]:
# data path
data_path = Path('D:/Data/2P/C57_O1M2/10022023/TSeries-10022023-1300-001')
print(f"Load data from {data_path}.")

# output path
export_path = Path('H:/Analysis/2P/C57_O1M2/10022023/run1/mesmerize/')
mc.set_parent_raw_data_path(export_path) 

# Create export_path if it doesn't exist
if not os.path.exists(export_path):
    os.makedirs(export_path)

# movie path
movie_path = Path.joinpath(export_path, 'cat_tiff_bt.tiff')

# batch path
batch_path = Path.joinpath(export_path, 'batch.pickle')

print(f"Export data to {export_path}.") 
print(f"Movie path: {movie_path}.") 
print(f"Batch path: {batch_path}.") 


### Concatenate the ome.tif files into a single multi-page tiff file

In [ ]:
# Using the concat_tif.py script to concatenate the tif files. If needed, install libtiff with: pip install pylibtiff
ct.concatenate_files([data_path], export_path)

### Create a new batch

This creates a new pandas `DataFrame` with the columns that are necessary for mesmerize. In mesmerize we call this the **batch DataFrame**. You can add additional columns relevant to your experiment, but do not modify columns used by mesmerize.

Note that when you create a DataFrame you will need to use `load_batch()` to load it later. You cannot use `create_batch()` to overwrite an existing batch DataFrame

In [ ]:
# create a new batch
# df = mc.create_batch(batch_path)
# to load existing batches use `load_batch()`
df = mc.load_batch(batch_path)
df

### Define parameters for search


In [ ]:
# copy the mcorr_params2 dict to make some changes
# new_params = deepcopy(mcorr_params2)
default_params = \
{
  'main':
    {
        'strides': [18, 18], #[48, 48],
        'overlaps': [24, 24],
        'max_shifts': [12, 12],
        'max_deviation_rigid': 6,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}

# new_params_1 = \
# {
#   'main':
#     {
#         'strides': [80, 80],
#         'overlaps': [40, 40],
#         'max_shifts': [12, 12],
#         'max_deviation_rigid': 6,
#         'border_nan': 'copy',
#         'pw_rigid': True,
#         'gSig_filt': None
#     },
# }

df.caiman.add_item(
  algo='mcorr',
  item_name=movie_path.stem,
  input_movie_path=movie_path,
  params=default_params
)
# df.caiman.add_item(
#   algo='mcorr',
#   item_name=movie_path.stem,
#   input_movie_path=movie_path,
#   params=new_params_1
# )

df


### Distinguishing parameter variants

We can see that there are many parameter variants, but it is not easy to see the differences in parameters between the rows that have the same `item_name`.

We can use the `caiman.get_params_diffs()` to see the unique parameters between rows with the same `item_name`

In [ ]:
diffs = df.caiman.get_params_diffs(algo="mcorr", item_name=df.iloc[0]["item_name"])
diffs

### Run multiple batch items.

`df.iterrows()` iterates through rows and returns the numerical index and row for each iteration

We can now see that there is one item in the DataFrame. What we called a "item" in `mesmerize-core` DataFrames is technically called a pandas `Series` or row.

In [ ]:
for i, row in df.iterrows():
    # If already processed, skip
    if row["outputs"] is not None:
        print(f"Skipping batch item {i}, id {row.uuid}, algo {row.algo}. Already run.") 
        continue
    process = row.caiman.run()
    
    # on Windows you MUST reload the batch dataframe after every iteration because it uses the `local` backend.
    # this is unnecessary on Linux & Mac
    # "DummyProcess" is used for local backend so this is automatic
    if process.__class__.__name__ == "DummyProcess":
        df = df.caiman.reload_from_disk()

### Outputs

Load the output information into the DataFrame

In [ ]:
df = df.caiman.reload_from_disk()
df

### Build your own visualizations using `fastplotlib`


In [ ]:
# first item is just the raw movie
movies = [df.iloc[0].caiman.get_input_movie()]

# subplot titles
subplot_names = ["raw"]

# add all the mcorr outputs to the list
for i, row in df.iterrows():
    # Select which row to display
    # if i == 2 or i == 3:
    # add to the list of movies to plot
    movies.append(row.mcorr.get_output())

    # subplot title to show dataframe index
    subplot_names.append(f"ix {i}")


# create the widget
mcorr_iw_multiple = fpl.ImageWidget(
    data=movies,  # list of movies
    window_funcs={"t": (np.mean, 1)}, # window functions as a kwarg, this is what the slider was used for in the ready-made viz
    grid_plot_kwargs={"size": (900, 900)},
    names=subplot_names,  # subplot names used for titles
    cmap="gray" #"gnuplot2"
)

mcorr_iw_multiple.show()

In [ ]:
mcorr_iw_multiple.close()

In [ ]:
# # make a list of rows we want to keep using the uuids
# rows_keep = [df.iloc[0].uuid]
# rows_keep

**Note:** On windows calling `remove_item()` will raise a `PermissionError` if you have the memmap file open. The workaround is to shutdown the current kernel and then use `df.caiman.remove_item()`. For example, you can keep another notebook that you use just for cleaning unwanted mcorr items.

There is currently no way to close a `numpy.memmap`: https://github.com/numpy/numpy/issues/13510

In [ ]:
# for i, row in df.iterrows():
#     if row.uuid not in rows_keep:
#         df.caiman.remove_item(row.uuid)

# df